# Energy Reconstruction -- True Energy vs Reco Charge

The 2D distribution of **true neutrino energy (y)** against **reco cluster
charge in ADC (x)**, one point per 1-to-1 matched true-reco pair -- the input to
a charge-to-energy calibration.

Built on the same population `Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb`
evaluates -- same input tree, same cuts, same 1-to-1 pairing -- with ONE
difference: `Apply_beam_window_cut` is **False** here, so every selected reco
cluster is available for matching rather than only the in-spill ones. That is
deliberate: with the beam-window cut on, the surviving reco population is
neutrino-dominated and `all_true_clusters/` comes out nearly identical to
`true_neutrino_clusters/`; with it off, cosmic tracks keep their own reco
clusters and the two populations genuinely differ. Set the flag back to True in
the configuration cell to restrict to the in-spill population -- the output
directory follows the flag, so the two never share a tree.

This notebook draws no completeness or purity plots -- it runs the pipeline only
as far as the pairing, which is what the plot needs.

**Which pairs**

- the **reco** side is a selected reco cluster, by construction: the pairing runs
  on the post-selection `clusters_reco`, so nothing that failed the cuts can appear
- **completeness > 80%** (`COMPLETENESS_THRESHOLD` below). A poorly reconstructed
  pair has most of its true energy in some other reco cluster or in none at all,
  so its charge cannot track its energy -- it would only smear the relation.
- the **true** side is split into two populations, drawn separately (below).
  Neutrino clusters are identified by `true_cluster_id >= 99990`, the id
  `reassign_cluster_ID_true_charge_light` gives interaction `nu_idx` -- an exact
  key, no spatial matching.

**Which true clusters** -- the same histogram over two populations, each into
its own subdirectory:

| subdirectory | pairs |
|---|---|
| `true_neutrino_clusters/` | pairs whose true side is a neutrino cluster |
| `all_true_clusters/` | every pair, neutrino and cosmic alike |

Both are selected from one record list, so the two can never end up built on
different pairings or different cuts.

**Which energy**: the true **cluster** energy, summed from the sed true points --
what the detector actually saw, and what every cut and completeness number in this
pipeline uses. mc.json's `Etot` (the *incident* neutrino energy) rides along in
the text table for reference but is not plotted: most of it may never be
deposited in the active volume, so plotting it would mix reconstruction with the
physics of how much energy the interaction left behind.

Each histogram carries a stats box with the pair count, the per-axis means and
the linear correlation, and a `*_vs_reco_charge.txt` table listing every pair
behind it. Empty bins are left blank rather than drawn as the colormap's zero.

At **job level** each histogram is also written to a `.root` file as a TH2D
named `true_energy_vs_reco_charge`, so a later analysis can pick up the binned
distribution without re-running this pipeline. Written with **uproot**, not
PyROOT (the ROOT installs on this machine conflict and PyROOT does not import);
the file is a normal ROOT file either way.

Plots are drawn at **event, file and job level** -- the same function at each,
so an event-level plot is a slice of the job-level one.

All drawing lives in `draw_energy_reconstruction.py` (this directory), which is
additive: it changes nothing in the existing pipeline modules and only consumes
records they already build. Output goes to
`EnergyReconstruction/multi_file_plots_charge_light_matching/EnergyReconstruction_{Before,After}TimeWindowCut/`,
the leaf chosen by `Apply_beam_window_cut`.


In [7]:
# Run scope -- same knobs as Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb.
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None   # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. The `files = N` knob above still takes the
# first N in lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) would skip every event.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# Decide which levels to draw
# ========================================================================
# Every level draws the SAME plots; they differ only in how many pairs are
# pooled into each. An event typically contributes ONE pair, so an event-level
# plot is a single point -- useful for checking one event, not for physics.
# Turn it off for a full-statistics run.
b_draw_event_level_plots = False   # one plot per event
b_draw_file_level_plots  = False   # one plot per file
b_draw_job_level_plots   = True   # one plot for the whole job


In [8]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in EnergyReconstruction/, one level below the repository
# root where the pipeline modules and the input trees are. Resolve both
# explicitly so the notebook runs whether Jupyter was started in this directory
# (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "EnergyReconstruction":
    NB_DIR = NB_DIR / "EnergyReconstruction"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Job started at: 2026-08-04 00:59:16
Notebook directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/EnergyReconstruction
Repository root:    /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction


In [9]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. This
# notebook runs the evaluation pipeline only as far as the 1-to-1 true-reco
# pairing, which is what the plots need; nothing past that (completeness / purity
# plotting, heatmaps, match multiplicity) is drawn here.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light, reassign_cluster_ID_reco,
    apply_energy_cutoff, apply_true_pointwise_energy_cutoff,
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from cluster_category import cluster_category
from completeness_purity_estimate import EvaluateCompleteness, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1
from metadata import (
    build_cluster_flash_metadata, build_img_cluster_flash_metadata,
    add_metadata_true_reco_pair_cluster, build_neutrino_vertex_records,
)
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US

# The new module for this notebook (EnergyReconstruction/draw_energy_reconstruction.py):
# the pair-record builder (which applies the neutrino + completeness selections)
# and the 2D drawers.
from draw_energy_reconstruction import (
    build_matched_pair_energy_records, draw_all_energy_reconstruction_plots,
    select_neutrino_pair_records, COMPLETENESS_THRESHOLD_DEFAULT,
)


In [10]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-img-global.json                       (reco clusters, imaging level)
#   file0/data/0/0-clustering-global.json                (reco clusters, post charge-light matching)
#   file0/data/0/0-sed-smear_readout.json                (true clusters)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)

# The DEAD-AREA-PREPROCESSED tree, produced once by preprocess_deadarea_cut.py.
# Its true-point files already have the dead-area cut applied, which is why
# Apply_deadarea_cut is False below. Read that script's docstring, or
# DEADAREA_PREPROCESSING.txt inside the tree, before switching this back to the
# raw tree: the two are NOT interchangeable.
PARENT_DIR = REPO_ROOT / "Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut"

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# ========================================================================
# THE SELECTION THIS NOTEBOOK IS ABOUT
# ========================================================================
# Minimum completeness for a 1-to-1 pair to enter the plots: 0.8 keeps pairs with
# MORE THAN 80% of the true cluster's energy reconstructed into the matched reco
# cluster. Below that, most of the energy is in some other reco cluster (or in
# none), so the pair's charge cannot be expected to track its energy. Set to 0
# to keep every pair.
COMPLETENESS_THRESHOLD = 0.8

# ========================================================================
# SELECTION PARAMETERS -- the same values as
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb, except for the
# beam-window cut (see below).
# ========================================================================
radius_completeness         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED: this format's
# point clouds are much sparser than the old imaging-based reconstruction --
# real neutrino clusters have been seen with as few as 13 points -- so the old
# threshold (200) would delete real signal clusters outright.
#
# min_cluster_energy IS applied: sed-smear's per-point 'e' field (MeV) is a
# genuine energy deposit, so the old threshold (100 MeV) carries over directly.
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_point_energy     = 0.02    # MeV per POINT (Apply_trueenergy_pointwise_cutoff below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

Apply_energy_cutoff                         = True
Apply_trueenergy_pointwise_cutoff           = True    # drop true POINTS below min_true_point_energy
Apply_min_true_points_cutoff                = False
Apply_min_reco_points_cutoff                = False
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# BEAM-WINDOW (time) CUT -- currently OFF, so EVERY selected reco cluster is
# available for matching, not just the in-spill ones.
#
# With it ON, only clusters whose bridged flash time lies inside
# [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US] = [0.33, 1.93] us survive, which is a
# neutrino-dominated population: a cosmic true cluster then has almost no
# well-matched reco cluster to pair with, so `all_true_clusters/` comes out
# nearly identical to `true_neutrino_clusters/`. With it OFF, cosmic tracks keep
# their own reco clusters and the two populations genuinely differ -- which is
# the point of drawing both.
#
# Set this back to True to restrict the plots to the in-spill population. The
# output directory below follows the flag, so before-cut and after-cut runs
# never land in the same tree.
Apply_beam_window_cut                       = False
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Set this True only if you point
# PARENT_DIR back at a raw tree.
Apply_deadarea_cut                          = False

# Wire-readout sensitive volume (detector geometry, unit: cm). Also the bounds
# for the vertex_in_volume flag carried onto each pair record.
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only -- there is no 2-view/3-view distinction in the
# charge-light format, so this is just a constant.
view = "combined"

# Output directory: inside EnergyReconstruction, so this notebook's output tree
# is self-contained and never shares a directory with the other notebooks' plots.
# The leaf name follows Apply_beam_window_cut -- the two settings describe
# different reco populations, and mixing their runs in one tree would make a
# timestamped directory the only clue as to which is which.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / (
    "EnergyReconstruction_AfterTimeWindowCut" if Apply_beam_window_cut
    else "EnergyReconstruction_BeforeTimeWindowCut")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

print("\nSelections for the plots:")
print(f"- 1-to-1 matched true-reco pairs, drawn for BOTH true populations "
      f"(true_neutrino_clusters/ and all_true_clusters/)")
print(f"- completeness > {COMPLETENESS_THRESHOLD:.0%}")

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-smear's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_beam_window_cut:
    print(f"- Beam window cut applied to RECO clusters only ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us flash time)")
else:
    print(f"- Beam window cut NOT applied -- every selected reco cluster is available for matching, "
          f"in-spill or not")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# ensure_data_extracted() only unzips if that file's data/ folder doesn't
# already exist, so re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


Configuration:
Parent directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
Plot base directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/EnergyReconstruction/multi_file_plots_charge_light_matching/EnergyReconstruction_BeforeTimeWindowCut
Files to process: all
Events to process: all

Selections for the plots:
- 1-to-1 matched true-reco pairs, drawn for BOTH true populations (true_neutrino_clusters/ and all_true_clusters/)
- completeness > 80%

Cuts applied:
- Energy cutoff applied (threshold 100 MeV, using sed-smear's per-point 'e' field)
- Wire readout sensitive xz plane cut applied
- Beam window cut NOT applied -- every selected reco cluster is available for matching, in-spill or not
- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)


In [11]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


Scanning parent directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
------------------------------------------------------------
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file1
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file10
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file2

In [12]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA energy reconstruction
# ============================================================================
# The selection chain below is a copy of
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb's, truncated after the
# 1-to-1 pairing: true side = sed-smear_readout grouped by REAL_CLUSTER_ID and
# reassigned to 99990+nu_idx (neutrino, one cluster per interaction) / avg-X
# (cosmic); reco side = clustering-global grouped by REAL_CLUSTER_ID (NOT
# cluster_id, which can merge physically distinct tracks), beam-window cut,
# fiducial cut, then relabelled by avg-X via reassign_cluster_ID_reco.
#
# The two plotted quantities both come straight off the pair record that
# pairing produces -- 'total_true_energy' (the true cluster's energy in MeV,
# summed from the sed points) and 'total_reco_charge' (the matched reco
# cluster's charge in ADC) -- so nothing is recomputed here that the evaluation
# does not already compute the same way.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

job_pair_energy_records = []   # one record per selected 1-to-1 pair (the plotted points)
job_pair_metadata_list  = []   # every 1-to-1 pair, before the neutrino/completeness selection
job_vertex_records      = []   # per true neutrino interaction (build_neutrino_vertex_records)
total_events_processed  = 0
total_files_processed   = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    file_output_dir = output_dir / input_file_name
    file_output_dir.mkdir(parents=True, exist_ok=True)

    file_pair_energy_records = []
    file_pair_metadata_list  = []
    file_vertex_records      = []

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key        = f"{input_file_name}_{evt}"
        event_output_dir = file_output_dir / f"event_{evt:03d}"
        event_output_dir.mkdir(parents=True, exist_ok=True)

        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        x_clu,  y_clu,  z_clu,  id_clu,  q_clu,  real_id_clu                       = result['clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # FLASH RECORDS: op.json flashes attached to img-global clusters, then
        # bridged onto clustering-global clusters by point charge ('q'). Used
        # here only to identify which reco clusters are in the beam window.
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}

        # ------------------------------------------------------------------
        # TRUE POINTS: sed-smear_readout in the standard 7-column shape
        # (energy = per-point 'e' in MeV -- the y axis of these plots comes from
        # summing this column; q_true = 'nu_idx', 0=cosmic, 1/2/...=which
        # neutrino interaction), reassigned to 99990+nu_idx (neutrino) / avg-X
        # (cosmic), then cut.
        # ------------------------------------------------------------------
        true_points = build_true_points_charge_light(
            x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited. Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        # POINT-wise first, so the cluster total the cluster cut tests is the
        # total of the points that survive.
        if Apply_trueenergy_pointwise_cutoff:
            true_points = apply_true_pointwise_energy_cutoff(true_points, min_true_point_energy)
        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points, x_min, x_max, y_min, y_max, z_min, z_max)
        if Apply_deadarea_cut:
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=event_output_dir, event=evt, file_name=input_file_name)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # RECO POINTS: clustering-global (post charge-light matching), grouped
        # by REAL_CLUSTER_ID. Beam-window cut FIRST -- clu_beam_window_ids lives
        # in the real_cluster_id namespace, which reassign_cluster_ID_reco
        # destroys, so filtering after it would match nothing.
        # ------------------------------------------------------------------
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))

        if Apply_beam_window_cut:
            n_reco_points_before_beam   = len(predicted_points)
            n_reco_clusters_before_beam = len(np.unique(predicted_points[:, 3])) if n_reco_points_before_beam else 0
            beam_ids_array   = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
            print(f"  Event {evt}: beam-window cut kept "
                  f"{len(clu_beam_window_ids)}/{n_reco_clusters_before_beam} reco clusters, "
                  f"{len(predicted_points)}/{n_reco_points_before_beam} reco points")

        if Apply_min_reco_points_cutoff:
            predicted_points = apply_min_reco_points_cutoff(predicted_points, min_reco_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            predicted_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_points, x_min, x_max, y_min, y_max, z_min, z_max)

        # An event can legitimately end up with NO in-spill reco cluster. It is
        # kept rather than skipped, but reassign_cluster_ID_reco cannot take an
        # empty array, so short-circuit to an empty dict -- that event simply
        # contributes no pair.
        if len(predicted_points) == 0:
            clusters_reco = {}
            print(f"  Event {evt}: no reco cluster survives the beam-window cut")
        else:
            predicted_points = reassign_cluster_ID_reco(predicted_points)
            clusters_reco    = GroupClustersByID(predicted_points)

        # ------------------------------------------------------------------
        # 1-TO-1 TRUE-RECO PAIRING (existing functions, unchanged). Completeness
        # and purity are computed because MatchTrueToReco1to1 needs them -- and
        # because the completeness is the selection this notebook applies.
        # ------------------------------------------------------------------
        cluster_category_results = cluster_category(clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)
        completeness_results       = EvaluateCompleteness(clusters_true, clusters_reco, event_key, radius_completeness, min_recopoints_threshold)
        purity_results           = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz, radius_purity_xy)

        event_matched_pairs      = MatchTrueToReco1to1(completeness_results, purity_results)
        event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
            event_matched_pairs, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        # True neutrino interaction vertices from mc.json, joined to their true
        # cluster by nu_idx (cluster_id = 99990+nu_idx, an exact key). These
        # carry mc.json's Etot -- the second y-axis variant -- and the
        # vertex_in_volume flag onto each pair record.
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # PAIR ENERGY RECORDS + DRAWING (draw_energy_reconstruction.py). The
        # neutrino and completeness selections are applied HERE, by the builder.
        # ------------------------------------------------------------------
        # neutrino_only=False: BOTH populations are drawn (neutrino pairs, and
        # every pair including cosmics), so the records keep the cosmic pairs and
        # draw_all_energy_reconstruction_plots selects the neutrino subset itself.
        # The completeness cut IS applied here, to both.
        event_pair_energy_records = build_matched_pair_energy_records(
            event_pair_metadata_list, vertex_records=event_vertex_records,
            completeness_threshold=COMPLETENESS_THRESHOLD, neutrino_only=False)

        if b_draw_event_level_plots:
            draw_all_energy_reconstruction_plots(
                event_pair_energy_records, event_output_dir,
                f"Event {evt}", f"event_{evt}", "Combined",
                file_name=input_file_name, completeness_threshold=COMPLETENESS_THRESHOLD)
            plt.close('all')

        # ------------------------------------------------------------------
        # AGGREGATE TO FILE AND JOB LEVEL
        # ------------------------------------------------------------------
        file_pair_energy_records.extend(event_pair_energy_records)
        file_pair_metadata_list.extend(event_pair_metadata_list)
        file_vertex_records.extend(event_vertex_records)

        job_pair_energy_records.extend(event_pair_energy_records)
        job_pair_metadata_list.extend(event_pair_metadata_list)
        job_vertex_records.extend(event_vertex_records)

        n_neutrino_pairs = sum(1 for p in event_pair_metadata_list
                               if float(p['true_cluster_id']) >= 99990)
        print(
            f"  Event {evt}: "
            f"1-to-1 pairs={len(event_pair_metadata_list)} (neutrino={n_neutrino_pairs}), "
            f"above {COMPLETENESS_THRESHOLD:.0%} completeness: all={len(event_pair_energy_records)}, "
            f"neutrino={len(select_neutrino_pair_records(event_pair_energy_records))}, "
            f"neutrino interactions in mc={len(event_vertex_records)}"
        )
        total_events_processed += 1

    # ========================================================================
    # FILE-LEVEL PLOTS: the same plots over every event in this file
    # ========================================================================
    if b_draw_file_level_plots:
        print(f"\n  FILE-LEVEL AGGREGATION ({input_file_name}): "
              f"{len(file_pair_metadata_list)} 1-to-1 pairs, "
              f"{len(file_pair_energy_records)} above the completeness cut")
        draw_all_energy_reconstruction_plots(
            file_pair_energy_records, file_output_dir / "file_summary",
            "File Level", "file", "Combined",
            file_name=input_file_name, completeness_threshold=COMPLETENESS_THRESHOLD)
        plt.close('all')

# ============================================================================
# JOB-LEVEL PLOTS: the same plots over every file and event
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
print(f"Total pairs above {COMPLETENESS_THRESHOLD:.0%} completeness: {len(job_pair_energy_records)} "
      f"(neutrino: {len(select_neutrino_pair_records(job_pair_energy_records))})")
print(f"{'='*70}")

job_output_dir = output_dir / "job_summary"
job_output_dir.mkdir(parents=True, exist_ok=True)

if b_draw_job_level_plots:
    # write_root=True here and nowhere else: the job-level histogram is the one
    # with the whole sample's statistics, and the one worth handing to a later
    # analysis.
    draw_all_energy_reconstruction_plots(
        job_pair_energy_records, job_output_dir, "Job Level", "job", "Combined",
        completeness_threshold=COMPLETENESS_THRESHOLD, write_root=True)
    plt.close('all')

# ============================================================================
# JOB SUMMARY TEXT FILE -- configuration, how many pairs each stage kept, and
# the summary numbers of the plotted distribution.
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

n_neutrino_pairs = sum(1 for p in job_pair_metadata_list if float(p['true_cluster_id']) >= 99990)

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY -- ENERGY RECONSTRUCTION (true energy vs reco charge)")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        summary_lines.append(f"Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("Cuts (identical to Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb):")
summary_lines.append(f"  energy cutoff:            {Apply_energy_cutoff} ({min_cluster_energy} MeV)")
summary_lines.append(f"  min true points cutoff:   {Apply_min_true_points_cutoff} ({min_true_points_cutoff})")
summary_lines.append(f"  min reco points cutoff:   {Apply_min_reco_points_cutoff} ({min_reco_points_cutoff})")
summary_lines.append(f"  wire readout volume cut:  {Apply_wire_readout_sensitive_xz_plane_cut}")
summary_lines.append(f"  beam window cut (reco):   {Apply_beam_window_cut} ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us)")
summary_lines.append(f"  dead area cut here:       {Apply_deadarea_cut} (applied upstream when False)")
summary_lines.append(f"  volume bounds: x [{x_min}, {x_max}], y [{y_min}, {y_max}], z [{z_min}, {z_max}] cm")
summary_lines.append("")
summary_lines.append("Plot selection:")
summary_lines.append(f"  completeness > {COMPLETENESS_THRESHOLD} (applied to both populations)")
summary_lines.append(f"  true_neutrino_clusters/: true side is a neutrino cluster (true_cluster_id >= 99990)")
summary_lines.append(f"  all_true_clusters/:      every pair, neutrino and cosmic alike")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
summary_lines.append(f"  of which the true side is a neutrino: {n_neutrino_pairs}")
summary_lines.append(f"Pairs above the completeness cut (PLOTTED):")
summary_lines.append(f"  all_true_clusters/:      {len(job_pair_energy_records)}")
summary_lines.append(f"  true_neutrino_clusters/: {len(select_neutrino_pair_records(job_pair_energy_records))}")
summary_lines.append(f"Total true neutrino interactions (mc.json): {len(job_vertex_records)}")
summary_lines.append("")

if job_pair_energy_records:
    charges  = np.array([r['reco_charge_ADC'] for r in job_pair_energy_records], dtype=float)
    energies = np.array([r['true_energy_MeV'] for r in job_pair_energy_records], dtype=float)
    mc_energies = np.array([r['mc_total_energy_MeV'] for r in job_pair_energy_records
                            if r['mc_total_energy_MeV'] is not None], dtype=float)
    summary_lines.append("Plotted distribution (job level, all_true_clusters):")
    summary_lines.append(f"  reco charge [ADC]:        mean {charges.mean():.4g}, "
                         f"min {charges.min():.4g}, max {charges.max():.4g}")
    summary_lines.append(f"  true cluster energy [MeV]: mean {energies.mean():.2f}, "
                         f"min {energies.min():.2f}, max {energies.max():.2f}")
    if charges.std() > 0 and energies.std() > 0 and len(charges) > 1:
        summary_lines.append(f"  correlation r (charge, true cluster energy): "
                             f"{np.corrcoef(charges, energies)[0, 1]:.4f}")
    if len(mc_energies) > 1:
        summary_lines.append(f"  mc neutrino energy Etot [MeV]: mean {mc_energies.mean():.2f}, "
                             f"min {mc_energies.min():.2f}, max {mc_energies.max():.2f} "
                             f"({len(mc_energies)} of {len(job_pair_energy_records)} pairs have one)")
    summary_lines.append("")

summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append("=" * 80)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")



Output directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/EnergyReconstruction/multi_file_plots_charge_light_matching/EnergyReconstruction_BeforeTimeWindowCut/combined_apa_20260804_005916


FILE 1/12: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Processing events 0 to 9


Found 7 matched pairs of true and reco clusters (1-to-1)
  Event 0: 1-to-1 pairs=7 (neutrino=0), above 80% completeness: all=6, neutrino=0, neutrino interactions in mc=1

Found 6 matched pairs of true and reco clusters (1-to-1)
  Event 1: 1-to-1 pairs=6 (neutrino=1), above 80% completeness: all=6, neutrino=1, neutrino interactions in mc=1

Found 8 matched pairs of true and reco clusters (1-to-1)
  Event 2: 1-to-1 pairs=8 (neutrino=2), above 80% completeness: all=8, neutrino=2, neutrino interactions in mc=2

Found 9 matched pairs of true and reco clusters (1-to-1)
  Event 3: 1-to-1 pairs=9 (neutr